# Silver Layer — Sellers Transformation

Transforms `bronze.sellers` into `silver.sellers`. Filters nulls, deduplicates by `seller_id`, cleans `seller_city` and `seller_state`, and validates with three DQ checks.

Write to silver schema.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../Utilities/silver_dq_checks

# Silver Layer — Data Quality Checks

This notebook defines reusable data quality check functions used across the Silver layer transformation notebooks.

Each function validates a specific property of a transformed DataFrame before it is written to a Silver Delta table. On success the function prints a PASSED message; on failure it raises a `ValueError` with details, stopping the pipeline.

## Checks Defined Here

* `check_not_null` — verifies that specified columns have no null or blank values
* `check_unique` — verifies that the given key (single or composite) has no duplicates
* `check_row_count` — verifies that the Silver row count falls within an expected percentage range of the Bronze source

## How To Use

Import these functions into a Silver transformation notebook by running:
​```
%run ../Utilities/silver_dq_checks
​```
Then call the functions on the transformed DataFrame before writing to Silver.

### check_not_null
Validates that the specified columns contain no null or blank/whitespace values.

### check_unique
Validates that the specified key columns produce unique rows. Supports single-column or composite keys.

### check_row_count
Validates that the Silver row count is within an acceptable percentage range of the Bronze source row count.


### check_event_sequence
Validates that timestamp columns appear in the expected chronological order. Skips rows where either timestamp in a pair is null. Raises on any violation.

### identify_event_sequence_violations
Adds one boolean column per consecutive timestamp pair to mark whether each row violates the expected order. Returns the annotated DataFrame without raising — callers use it to split clean rows from violations.

### check_value_range
Validates that values in specified columns meet a minimum threshold rule (e.g. `price > 0`, `freight_value >= 0`). Each rule is a dict specifying column, min_value, and inclusivity. Raises on any violation, reporting all rules that failed.

In [0]:
catalog_name = 'retail_lakehouse'
bronze_schema = 'bronze'
silver_schema = 'silver'
quarantine_schema = 'quarantine'

### Transformations

In [0]:
df_sellers = spark.read.table(f'{catalog_name}.{bronze_schema}.sellers')

transformed_sellers = df_sellers.filter(col('seller_id').isNotNull())\
    .dropDuplicates(['seller_id'])\
        .withColumn('seller_city',lower(trim(col('seller_city'))))\
            .withColumn('seller_state',lower(trim(col('seller_state'))))\
                .withColumn('silver_processed_at',current_timestamp())

### Data Quality Checks

In [0]:
check_not_null(transformed_sellers,['seller_id','seller_zip_code_prefix','seller_city','seller_state'])
check_unique(transformed_sellers,['seller_id'])
check_row_count(transformed_sellers,f'{catalog_name}.{bronze_schema}.sellers',85,100)


check_not_null: PASSED
check_unique: PASSED
check_row_count: PASSED


## Writing transformed dataframe to silver schema

In [0]:
transformed_sellers.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.{silver_schema}.sellers')